In [ ]:
%load_ext autoreload
%autoreload 2

import os

from pymongo import MongoClient
import gridfs
from dotenv import load_dotenv

load_dotenv()
MONGODB_URI = os.getenv("MONGODB_URI")
if not MONGODB_URI:
    raise ValueError("MONGODB_URI is not set")
client = MongoClient(MONGODB_URI)
db = client['ebl']
files_collection = db['photos.files']
chunks_collection = db['photos.chunks']
fs = gridfs.GridFS(db, collection='photos')

photo_count = files_collection.count_documents({})
print(f"Total photos: {photo_count}")

cursor = files_collection.find({})

photos = list(cursor)

photo_count = len(photos)
print(f"Total photos: {photo_count}")

# chunks = list(chunks_collection.find({}))

# print(f"Total data chunks: {len(chunks)}")

filenames = fs.list()
print(f"Total files: {len(filenames)}")

In [ ]:
import random
# shuffle the photos
print("The first 5 photos before shuffling:")
for photo in photos[:5]:
    print(photo['filename'])

random.shuffle(photos)
print("The first 5 photos after shuffling:")
for photo in photos[:5]:
    print(photo['filename'])

photos_to_check = photos[:30]

special_list = ["HS.2086.jpg", "ND.5437.jpg", "BM.98496.jpg"]
# search for the special photos in the shuffled list
for photo in photos:
    if photo['filename'] in special_list:
        photos_to_check.insert(0, photo)

In [ ]:
from tqdm import tqdm
from time import sleep

# Create a fresh cursor
cursor = files_collection.find({})

# Use tqdm with the cursor
for doc in tqdm(cursor, total=photo_count):
    file_id = doc['_id']
    filename = doc.get('filename', 'unknown')
    # sleep(0.0001)  # Your processing logic here
    # Your processing logic here

In [ ]:
import os
import io   
import shutil

from PIL import Image, UnidentifiedImageError
import cv2

Image.MAX_IMAGE_PIXELS = None  # Disable DecompressionBombError


if os.path.exists("division_checking_photos"):
    shutil.rmtree("division_checking_photos")
os.makedirs("division_checking_photos", exist_ok=True)

for i, photo in enumerate(photos_to_check):
    
    file_id = photo['_id']
    filename = photo['filename']
    if filename in special_list:
        filename = "special_{}".format(filename)
    grid_out = fs.get(file_id)
    data = grid_out.read()
    print(f"Processing photo {i+1}: {filename}, Size: {len(data)} bytes")
    try:
        image = Image.open(io.BytesIO(data))
        image.save(f"{filename}")
    except UnidentifiedImageError as e:
        print(f"Skipping file {filename}: Image cannot be identified - {str(e)}")
        continue  # Skip to the next file if the image is invalid
    
    image = cv2.imread(f"{filename}")

    from divide_photos import divide_tablet_photo
    cropped_images, _ = divide_tablet_photo(
        image,
        max_objects=30,
        visualize=True,  # Generate visualization result
        output_path="division_checking_photos/{}".format(filename),
        force_output_result=True,
        return_masks=True
    )

    os.remove(f"{filename}")  # Clean up the saved image file after processing